In [4]:
import sys
sys.path.append('../utils_function/')
sys.path.append(r'/data/zyjin/common_func/')
import utils_function as uf
import common_functions as cf
import pandas as pd
import numpy as np
import os
from collections import defaultdict

In [5]:
def get_optimal_params(performance_dict):
    """
    从性能字典中找出平均相关性最高的参数
    
    参数:
    performance_dict (dict): 包含不同参数对应相关性列表的字典
    
    返回:
    float: 最优参数值
    """
    best_avg_corr = -float('inf')  # 初始化为负无穷

    for param, corr_list in performance_dict.items():
        avg_corr = sum(corr_list) / len(corr_list)
        std_corr = np.std(corr_list)
        # print(f"参数: {param}, 平均相关性: {avg_corr:.4f}, 标准差: {std_corr:.4f}")
        
        # 更新最优参数
        if avg_corr > best_avg_corr:
            best_avg_corr = avg_corr
            best_param = param

    return best_param

In [6]:
# animal_list = ['mouse', 'marmoset', 'macaque', 'human', 'marmoset']
# dataset_key_list = ['gamanut', 'majka', 'markov40', 'lariviere', 'V4']
animal_list = ['mouse', 'marmoset', 'macaque', 'human']
dataset_key_list = ['gamanut', 'majka', 'markov40', 'lariviere']
performance_dict = {}
optimal_params_dict = {}
for animal, dataset_key in zip(animal_list, dataset_key_list):
    print(f'Running for {animal} and {dataset_key}')

    performance_dict_this_animal = defaultdict(list)
    performance_dict_this_animal_with_seed = defaultdict(list)

    model_container = cf.MetaModelContainer(uf.AxonGrowthModel, cf.pj('../../results/para_search_betzel_like', f'{animal}_{dataset_key}'))

    for model in model_container:
        performance_dict_this_animal[(model.Lambda, model.gamma)].append(model.get_data_model_correlation())
        performance_dict_this_animal_with_seed[(model.Lambda, model.gamma, model.the_seed)].append(model.get_data_model_correlation())
    
    performance_dict[f'{animal}_{dataset_key}'] = performance_dict_this_animal
    # get_optimal_params(performance_dict_this_animal)
    best_Lambda, best_gamma, best_seed = get_optimal_params(performance_dict_this_animal_with_seed)
    optimal_params_dict[f'{animal}_{dataset_key}'] = {'Lambda': best_Lambda, 'gamma': best_gamma, 'the_seed': best_seed}
    print(performance_dict_this_animal_with_seed)
    print(f'Optimal lambda for {animal}_{dataset_key}: {optimal_params_dict[f"{animal}_{dataset_key}"]}')
    print(f'Optimal average correlation: {max([sum(v)/len(v) for v in performance_dict_this_animal_with_seed.values()])}')

Running for mouse and gamanut
defaultdict(<class 'list'>, {(0.19995164268532023, -1.1111111111111112, 4): [-0.06089297032350531], (0.4998065707412809, 1.5555555555555554, 5): [0.5512158653453022], (0.6997098561119213, -0.6666666666666667, 5): [-0.060820895965547614], (0.8996131414825617, 1.5555555555555554, 4): [0.488149333764768], (0.1, -2.0, 4): [-0.060893060792172056], (0.8996131414825617, -2.0, 5): [-0.06089306079148943], (0.6997098561119213, 2.0, 1): [0.5235225910073472], (0.5997582134266011, -0.6666666666666667, 4): [-0.060819027941350044], (0.6997098561119213, 1.1111111111111107, 5): [0.5324179373021879], (0.8996131414825617, 0.6666666666666665, 2): [0.4866466942938268], (0.5997582134266011, 0.6666666666666665, 3): [0.5426301434695693], (0.6997098561119213, -0.22222222222222232, 4): [-0.02755497513895765], (0.9995647841678819, 0.22222222222222232, 3): [0.47354030756210735], (0.19995164268532023, -2.0, 1): [-0.06089306079188234], (0.1, 0.22222222222222232, 2): [0.4903404564555881

In [ ]:
# if cf.check_all_file_exist_with_any_extension('../../results/optimal_params_betzel_like/optimal_params_betzel_like'):
#     print('already saved, please check whether you want to overwrite it')
# else:
cf.save_dict(optimal_params_dict, '../../results/optimal_params_betzel_like/optimal_params_betzel_like')